# Apple Health Export XML Analysis

## XML structure
The local `export.xml` file has this high-level structure:

```text
HealthData (1)
+-- ExportDate (1)
+-- Me (1)
+-- Record (2,922,801)
|   +-- MetadataEntry (1,777,509)
|   `-- HeartRateVariabilityMetadataList (7,324)
|       `-- InstantaneousBeatsPerMinute (398,933)
+-- ActivitySummary (1,568)
`-- Workout (442)
    +-- MetadataEntry (4,338)
    +-- WorkoutStatistics (1,383)
    +-- WorkoutEvent (1,243)
    `-- WorkoutRoute (98)
        +-- MetadataEntry (196)
        `-- FileReference (98)
```

Main analysis branches:

- `Record`: individual health measurements such as steps, heart rate, sleep, energy, and distance.
- `ActivitySummary`: daily activity-ring totals and goals.
- `Workout`: workout sessions with nested statistics, events, and route references.


In [4]:
from __future__ import annotations
from pathlib import Path
# Apple Health XML helpers merged from apple_health_xml.py.

from collections import Counter

import pandas as pd


DATE_COLUMNS = {
    "creationDate",
    "startDate",
    "endDate",
    "date",
    "dateComponents",
}

NUMERIC_COLUMNS = {
    "value",
    "duration",
    "totalDistance",
    "totalEnergyBurned",
    "activeEnergyBurned",
    "activeEnergyBurnedGoal",
    "appleMoveTime",
    "appleMoveTimeGoal",
    "appleExerciseTime",
    "appleExerciseTimeGoal",
    "appleStandHours",
    "appleStandHoursGoal",
}

PREFIXES = (
    "HKQuantityTypeIdentifier",
    "HKCategoryTypeIdentifier",
    "HKWorkoutActivityType",
    "HKDataType",
    "HKCharacteristicTypeIdentifier",
)


def short_health_name(value: object) -> object:
    """Remove common HealthKit prefixes while preserving non-string values."""
    if not isinstance(value, str):
        return value

    for prefix in PREFIXES:
        if value.startswith(prefix):
            return value.removeprefix(prefix)
    return value





In [5]:
EXPORT_XML = next(path for path in [Path("../../health_data/export.xml")])

## Inspect available Apple Health record types

## Convert records to DataFrames

In [6]:
from analysis import apple_health_xml_to_df
# Load one metric while exploring. Remove type_filter to load all ~millions of records.
steps_df = apple_health_xml_to_df(
    EXPORT_XML,
    tag="Record",
    type_filter="HKQuantityTypeIdentifierStepCount",
)
steps_df.head()

,element_tag,type,sourceName,sourceVersion,device,unit,creationDate,startDate,endDate,value,parent_HealthData_locale,value_numeric
0,Record,HKQuantityTypeIdentifierStepCount,Claudio’s Big Brick,15.0.1,"<<HKDevice: 0xc1d5914a0>, name:iPhone, manufac...",count,2021-11-02 05:49:39+10:00,2021-11-02 05:38:31+10:00,2021-11-02 05:48:23+10:00,20,en_DK,20
1,Record,HKQuantityTypeIdentifierStepCount,Claudio’s Big Brick,15.0.1,"<<HKDevice: 0xc1d5914a0>, name:iPhone, manufac...",count,2021-11-02 06:11:41+10:00,2021-11-02 06:00:38+10:00,2021-11-02 06:00:48+10:00,17,en_DK,17
2,Record,HKQuantityTypeIdentifierStepCount,Claudio’s Big Brick,15.0.1,"<<HKDevice: 0xc1d5914a0>, name:iPhone, manufac...",count,2021-11-02 06:23:17+10:00,2021-11-02 06:12:14+10:00,2021-11-02 06:14:32+10:00,17,en_DK,17
3,Record,HKQuantityTypeIdentifierStepCount,Claudio’s Big Brick,15.0.1,"<<HKDevice: 0xc1d5914a0>, name:iPhone, manufac...",count,2021-11-02 08:12:22+10:00,2021-11-02 08:09:07+10:00,2021-11-02 08:09:27+10:00,22,en_DK,22
4,Record,HKQuantityTypeIdentifierStepCount,Claudio’s Big Brick,15.0.1,"<<HKDevice: 0xc1d5914a0>, name:iPhone, manufac...",count,2021-11-02 21:23:15+10:00,2021-11-02 21:17:39+10:00,2021-11-02 21:17:44+10:00,9,en_DK,9


## Convert workouts and daily activity summaries

In [ ]:
hrv_beats_df = apple_health_xml_to_df(
    EXPORT_XML,
    tag="InstantaneousBeatsPerMinute",
    ancestor_type_filter="HKQuantityTypeIdentifierHeartRateVariabilitySDNN",
)

In [22]:
reduced_hrv_beats_df = hrv_beats_df.drop(columns=["parent_Record_creationDate","parent_Record_sourceVersion","parent_Record_sourceName","parent_HealthData_locale","parent_Record_device","parent_Record_type"])

reduced_hrv_beats_df

,element_tag,bpm,time,parent_Record_unit,parent_Record_startDate,parent_Record_endDate,parent_Record_value,bpm_numeric
0,InstantaneousBeatsPerMinute,86,"11.28.15,14",ms,2022-01-09 10:28:14+10:00,2022-01-09 10:29:09+10:00,72.3623,86
1,InstantaneousBeatsPerMinute,88,"11.28.15,82",ms,2022-01-09 10:28:14+10:00,2022-01-09 10:29:09+10:00,72.3623,88
2,InstantaneousBeatsPerMinute,88,"11.28.16,50",ms,2022-01-09 10:28:14+10:00,2022-01-09 10:29:09+10:00,72.3623,88
3,InstantaneousBeatsPerMinute,88,"11.28.17,18",ms,2022-01-09 10:28:14+10:00,2022-01-09 10:29:09+10:00,72.3623,88
4,InstantaneousBeatsPerMinute,86,"11.28.17,87",ms,2022-01-09 10:28:14+10:00,2022-01-09 10:29:09+10:00,72.3623,86
...,...,...,...,...,...,...,...,...
398928,InstantaneousBeatsPerMinute,69,"17.01.51,81",ms,2026-05-23 17:00:55+10:00,2026-05-23 17:01:54+10:00,69.1693,69
398929,InstantaneousBeatsPerMinute,64,"17.01.52,75",ms,2026-05-23 17:00:55+10:00,2026-05-23 17:01:54+10:00,69.1693,64
398930,InstantaneousBeatsPerMinute,74,"17.01.53,56",ms,2026-05-23 17:00:55+10:00,2026-05-23 17:01:54+10:00,69.1693,74
398931,InstantaneousBeatsPerMinute,86,"17.01.54,26",ms,2026-05-23 17:00:55+10:00,2026-05-23 17:01:54+10:00,69.1693,86
